In [0]:
%pip install xgboost==3.0.5
dbutils.library.restartPython()

### Jointure entre les 6 dataframe

In [0]:
%sql
CREATE OR REPLACE TABLE NBOSportScoring (
SELECT i.*
      , t.TV_DURATION_CULTURE_L3M_SECONDES
      , t.TV_DURATION_CULTURE_L3M_DAYS
      , t.TV_RANK_CULTURE_L3M
      , t.TV_DURATION_CYCLING_L3M_SECONDES
      , t.TV_DURATION_CYCLING_L3M_DAYS
      , t.TV_RANK_CYCLING_L3M
      , t.TV_DURATION_DOCUMENTARIES_L3M_SECONDES
      , t.TV_DURATION_DOCUMENTARIES_L3M_DAYS
      , t.TV_RANK_DOCUMENTARIES_L3M
      , t.TV_DURATION_FOOTBALL_L3M_SECONDES
      , t.TV_DURATION_FOOTBALL_L3M_DAYS
      , t.TV_RANK_FOOTBALL_L3M
      , t.TV_DURATION_MOVIES_L3M_SECONDES
      , t.TV_DURATION_MOVIES_L3M_DAYS
      , t.TV_RANK_MOVIES_L3M
      , m.AMOUNT_OUT_OF_BUNDLE_L3M
      , m.NB_DAYS_OUT_OF_BUNDLE_L3M
      , s.AVG_CUST_AGE
      , s.AVG_NB_DAYS_INTERNET_LINE
      , s.AVG_NB_DAYS_TV_LINE
      , s.AVG_NB_MOBILE_SUBS
      , j.AMOUNT_OF_CONTACT_ADMINISTRATIVE_ISSUES_L3M
      , j.AMOUNT_OF_CONTACT_COMPLAINTS_L3M
      , j.JOURNEY_DURATION_ADMINISTRATIVE_ISSUES_L3M_DAYS
      , j.JOURNEY_DURATION_COMPLAINTS_L3M_DAYS
      , j.JOURNEY_DURATION_TECHNICAL_ISSUES_L3M_DAYS
      , j.TOTAL_NUMBER_OF_CONTACTS_L3M
      , j.TOTAL_NUMBER_OF_JOURNEYS_L3M
FROM nboTVAggregateM_1 AS T
FULL JOIN nboInternetAggregateM_1 AS I
  ON T.CUST_NUM = I.CUST_NUM AND T.YEAR = I.YEAR AND T.MONTH = I.MONTH

LEFT JOIN nboMobileAggregateM_1 AS M
  ON COALESCE(T.CUST_NUM, I.CUST_NUM) = M.CUST_NUM
 AND COALESCE(T.YEAR, I.YEAR) = M.YEAR
 AND COALESCE(T.MONTH, I.MONTH) = M.MONTH

LEFT JOIN nboSocioDemoAggregateM_1 AS S
  ON COALESCE(T.CUST_NUM, I.CUST_NUM) = S.CUST_NUM
 AND COALESCE(T.YEAR, I.YEAR) = S.YEAR
 AND COALESCE(T.MONTH, I.MONTH) = S.MONTH

LEFT JOIN nboJourneysAggregateM_1 AS J
  ON COALESCE(T.CUST_NUM, I.CUST_NUM) = J.CUST_NUM
 AND COALESCE(T.YEAR, I.YEAR) = J.YEAR
 AND COALESCE(T.MONTH, I.MONTH) = J.MONTH
);

In [0]:
df = spark.table("NBOSportScoring")

In [0]:
df.columns

%md
### Remplacement des valeurs nulles par des '-1'

In [0]:
df_scoring = df.fillna(-1)

### Prédiction données de scoring

In [0]:
%pip install flask

In [0]:
import mlflow
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

model_uri = "models:/sport_model@last"

def cast_features_to_double(df, feature_cols):
    """
    Force toutes les colonnes de features en DoubleType
    pour être compatibles avec la signature MLflow.
    """
    for col_name in feature_cols:
        df = df.withColumn(col_name, F.col(col_name).cast(DoubleType()))
    return df

# Créer un UDF Spark
model_udf = mlflow.pyfunc.spark_udf(
    spark,
    model_uri=model_uri,
    result_type=DoubleType(),
    env_manager="local"
)

# Colonnes à garder pour info mais pas pour le scoring
info_cols = ['CUST_NUM', 'MONTH', 'YEAR']

# Colonnes de features pour le modèle
feature_cols = [col for col in df_scoring.columns if col not in info_cols]

# Cast des features
df_scoring = cast_features_to_double(df_scoring, feature_cols)

# Application du modèle
df_scoring = df_scoring.withColumn(
    "y_proba", 
    model_udf(*[F.col(c) for c in feature_cols])
)

# Ajout de la prédiction binaire
df_scoring = df_scoring.withColumn(
    "PREDICTION_SPORT", 
    (F.col("y_proba") >= F.lit(0.16)).cast("int")
)

### Conversion du dataframe en delta table + test

In [0]:
%sql
DROP TABLE IF EXISTS nboSportScoringPred;

In [0]:
# Sauver le dataframe au format delta table natif de Databricks
df_scoring.write.format("delta").saveAsTable("nboSportScoringPred")

In [0]:
%sql
-- Test ouverture en SQL de la delta table
SELECT * 
FROM nboSportScoringPred;